In [1]:
import os, glob, re
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Shared helpers
DATA_DIR = 'data/'

def find_file(pattern):
    matches = glob.glob(os.path.join(DATA_DIR, '**', pattern), recursive=True)
    if not matches:
        raise FileNotFoundError(f'Could not find {pattern} under {DATA_DIR}')
    return sorted(matches, key=len)[0]

def find_file_optional(pattern):
    matches = glob.glob(os.path.join(DATA_DIR, '**', pattern), recursive=True)
    if not matches:
        return None
    return sorted(matches, key=len)[0]

def parse_seed(seed_str: str):
    region = seed_str[0]
    seed_num = int(seed_str[1:3])
    play = seed_str[3:] if len(seed_str) > 3 else ''
    return region, seed_num, play

def make_team_game_rows(reg_df: pd.DataFrame, diff_suffix: str = 'dif') -> pd.DataFrame:
    df = reg_df.copy()

    w = pd.DataFrame({
        'Season': df['Season'].values,
        'DayNum': df['DayNum'].values,
        'TeamID': df['WTeamID'].values,
        'OppID': df['LTeamID'].values,
        'IsWin': 1,
        'WLoc': df['WLoc'].values if 'WLoc' in df.columns else 'N',
    })

    l = pd.DataFrame({
        'Season': df['Season'].values,
        'DayNum': df['DayNum'].values,
        'TeamID': df['LTeamID'].values,
        'OppID': df['WTeamID'].values,
        'IsWin': 0,
        'WLoc': df['WLoc'].values if 'WLoc' in df.columns else 'N',
    })

    EXCLUDE_STATS = {'TeamID', 'OppID', 'Season', 'DayNum', 'NumOT', 'WLoc'}
    stat_names = []
    for c in df.columns:
        if re.match(r'^[WL][A-Z].*', c):
            base = c[1:]
            if base not in EXCLUDE_STATS:
                stat_names.append(base)
    stat_names = sorted(set(stat_names))

    for s in stat_names:
        w_team = 'W' + s
        w_opp  = 'L' + s
        l_team = 'L' + s
        l_opp  = 'W' + s

        if w_team in df.columns and w_opp in df.columns:
            w[f'T_{s}'] = df[w_team].values
            w[f'O_{s}'] = df[w_opp].values
        if l_team in df.columns and l_opp in df.columns:
            l[f'T_{s}'] = df[l_team].values
            l[f'O_{s}'] = df[l_opp].values

    out = pd.concat([w, l], axis=0, ignore_index=True)

    def loc_from_team_pov(is_win, wloc):
        if wloc not in ('H', 'A', 'N'):
            return 'N'
        if wloc == 'N':
            return 'N'
        if is_win == 1:
            return wloc
        return 'A' if wloc == 'H' else 'H'

    out['Loc'] = [loc_from_team_pov(w, wl) for w, wl in zip(out['IsWin'], out['WLoc'])]

    if 'T_Score' in out.columns and 'O_Score' in out.columns:
        out['Margin'] = pd.to_numeric(out['T_Score'], errors='coerce') - pd.to_numeric(out['O_Score'], errors='coerce')
    else:
        score_candidates = [c for c in out.columns if c.lower() in ('t_score','t_pts','t_points')]
        opp_candidates = [c for c in out.columns if c.lower() in ('o_score','o_pts','o_points')]
        if score_candidates and opp_candidates:
            out['Margin'] = pd.to_numeric(out[score_candidates[0]], errors='coerce') - pd.to_numeric(out[opp_candidates[0]], errors='coerce')
        else:
            out['Margin'] = 0.0

    out['Loc_H'] = (out['Loc'] == 'H').astype(np.int8)
    out['Loc_A'] = (out['Loc'] == 'A').astype(np.int8)
    out['Loc_N'] = (out['Loc'] == 'N').astype(np.int8)

    for s in stat_names:
        tcol = f'T_{s}'
        ocol = f'O_{s}'
        dif_col = f'{s}_{diff_suffix}'
        if tcol in out.columns and ocol in out.columns:
            out[tcol] = pd.to_numeric(out[tcol], errors='coerce')
            out[ocol] = pd.to_numeric(out[ocol], errors='coerce')
            out[dif_col] = out[tcol] - out[ocol]

    numeric_and_keys = ['Season', 'DayNum', 'TeamID', 'OppID', 'IsWin', 'WLoc', 'Loc', 'Loc_H', 'Loc_A', 'Loc_N']
    remaining = [c for c in out.columns if (pd.api.types.is_numeric_dtype(out[c]) or c in numeric_and_keys)]
    out = out[remaining]

    return out.sort_values(['Season','TeamID','DayNum']).reset_index(drop=True)

def add_possession_efficiency_features(df):
    out = df.copy()
    required = [
        'T_FGA', 'T_OR', 'T_TO', 'T_FTA', 'T_Score',
        'O_FGA', 'O_OR', 'O_TO', 'O_FTA', 'O_Score'
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f'Missing required columns for possessions: {missing}')

    out['T_Poss'] = out['T_FGA'] - out['T_OR'] + out['T_TO'] + 0.475 * out['T_FTA']
    out['O_Poss'] = out['O_FGA'] - out['O_OR'] + out['O_TO'] + 0.475 * out['O_FTA']
    out['Poss'] = 0.5 * (out['T_Poss'] + out['O_Poss']).clip(lower=1.0)

    out['T_OffEff'] = 100.0 * out['T_Score'] / out['Poss']
    out['T_DefEff'] = 100.0 * out['O_Score'] / out['Poss']
    out['T_NetEff'] = out['T_OffEff'] - out['T_DefEff']

    out['O_OffEff'] = 100.0 * out['O_Score'] / out['Poss']
    out['O_DefEff'] = 100.0 * out['T_Score'] / out['Poss']
    out['O_NetEff'] = out['O_OffEff'] - out['O_DefEff']

    out['OffEff_dif'] = out['T_OffEff'] - out['O_OffEff']
    out['DefEff_dif'] = out['T_DefEff'] - out['O_DefEff']
    out['NetEff_dif'] = out['T_NetEff'] - out['O_NetEff']

    return out

def add_rolling_neteff_features_team_and_opp(df, windows=(5, 10), feature_col='T_NetEff'):
    out = df.copy()
    out = out.sort_values(['Season', 'TeamID', 'DayNum'], kind='mergesort')
    if feature_col not in out.columns:
        raise ValueError(f"Missing feature_col='{feature_col}'. Did you run add_possession_efficiency_features first?")

    grpT = out.groupby(['Season', 'TeamID'], sort=False)
    for w in windows:
        out[f'T_NetEff_Mean_{w}'] = grpT[feature_col].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
        out[f'T_NetEff_Std_{w}'] = grpT[feature_col].transform(lambda s: s.shift(1).rolling(w, min_periods=1).std())

    for c in out.columns:
        if c.startswith('T_NetEff_Std_'):
            out[c] = out[c].fillna(0.0)

    base_keys = ['Season', 'DayNum', 'TeamID', 'OppID']
    tmp = out[base_keys + [feature_col]].copy()
    tmp = tmp.rename(columns={'OppID': 'TeamID', 'TeamID': 'OrigTeamID'})
    tmp = tmp.sort_values(['Season', 'TeamID', 'DayNum'], kind='mergesort')
    grpO = tmp.groupby(['Season', 'TeamID'], sort=False)

    opp_roll = tmp[['Season', 'DayNum', 'OrigTeamID', 'TeamID']].copy()
    opp_roll = opp_roll.rename(columns={'OrigTeamID': 'TeamID', 'TeamID': 'OppID'})

    for w in windows:
        opp_roll[f'O_NetEff_Mean_{w}'] = grpO[feature_col].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean()).values
        opp_roll[f'O_NetEff_Std_{w}'] = grpO[feature_col].transform(lambda s: s.shift(1).rolling(w, min_periods=1).std()).values

    for c in opp_roll.columns:
        if c.startswith('O_NetEff_Std_'):
            opp_roll[c] = opp_roll[c].fillna(0.0)

    out = out.merge(opp_roll, on=['Season', 'DayNum', 'TeamID', 'OppID'], how='left')

    for w in windows:
        out[f'NetEff_Mean_dif_{w}'] = out[f'T_NetEff_Mean_{w}'] - out[f'O_NetEff_Mean_{w}']
        out[f'NetEff_Std_dif_{w}']  = out[f'T_NetEff_Std_{w}']  - out[f'O_NetEff_Std_{w}']

    return out

def add_massey_asof_groupwise(team_games_df, massey_df, system_name, id_col='TeamID', prefix='T'):
    left = team_games_df.copy()
    left['Season'] = left['Season'].astype(np.int64)
    left[id_col] = left[id_col].astype(np.int64)
    left['DayNum'] = left['DayNum'].astype(np.int64)

    right = massey_df[massey_df['SystemName'] == system_name][['Season','TeamID','RankingDayNum','OrdinalRank']].copy()
    right['Season'] = right['Season'].astype(np.int64)
    right['TeamID'] = right['TeamID'].astype(np.int64)
    right['RankingDayNum'] = right['RankingDayNum'].astype(np.int64)
    right['Rank'] = right['OrdinalRank'].astype(np.float32)
    right['RatingNegRank'] = -right['Rank']

    right = (right.sort_values(['Season','TeamID','RankingDayNum'], kind='mergesort')
                  .drop_duplicates(['Season','TeamID','RankingDayNum'], keep='last'))

    out_parts = []
    right_groups = {k: g for k, g in right.groupby(['Season','TeamID'], sort=False)}

    for k, gL in left.groupby(['Season', id_col], sort=False):
        gL = gL.sort_values('DayNum', kind='mergesort')
        gR = right_groups.get((k[0], k[1]), None)
        if gR is None or len(gR) == 0:
            gL[f'{system_name}_{prefix}_Rating'] = -255.0
            out_parts.append(gL)
            continue
        gR = gR.sort_values('RankingDayNum', kind='mergesort')
        merged = pd.merge_asof(
            gL,
            gR[['RankingDayNum','Rank','RatingNegRank']],
            left_on='DayNum',
            right_on='RankingDayNum',
            direction='backward',
            allow_exact_matches=True
        )
        gL[f'{system_name}_{prefix}_Rating'] = merged['RatingNegRank'].values
        out_parts.append(gL)

    return pd.concat(out_parts, ignore_index=True)

def add_team_and_opp_massey(team_games_df, massey_df, system_name):
    out = add_massey_asof_groupwise(team_games_df, massey_df, system_name, id_col='TeamID', prefix='T')
    out = add_massey_asof_groupwise(out, massey_df, system_name, id_col='OppID', prefix='O')
    out[f'{system_name}_Rating_Diff'] = out[f'{system_name}_T_Rating'] - out[f'{system_name}_O_Rating']
    return out

TOURNEY_CUTOFF_DAY = 133
def build_pre_tourney_context(reg_df, cutoff_day=133):
    df = reg_df.copy()
    df = df[df['DayNum'] <= cutoff_day]
    w = df[['Season','DayNum','WTeamID','LTeamID']].copy()
    w['TeamID'] = w['WTeamID']
    w['OppID'] = w['LTeamID']
    w['Win'] = 1
    l = df[['Season','DayNum','WTeamID','LTeamID']].copy()
    l['TeamID'] = l['LTeamID']
    l['OppID'] = l['WTeamID']
    l['Win'] = 0
    games = pd.concat([
        w[['Season','DayNum','TeamID','OppID','Win']],
        l[['Season','DayNum','TeamID','OppID','Win']],
    ], ignore_index=True)
    agg = (
        games.groupby(['Season','TeamID']).agg(Wins=('Win','sum'), Games=('Win','count')).reset_index()
    )
    agg['Losses'] = agg['Games'] - agg['Wins']
    agg['WinPct'] = agg['Wins'] / agg['Games']
    games = games.sort_values(['Season','TeamID','DayNum'])
    last10 = (
        games.groupby(['Season','TeamID'])['Win'].apply(lambda s: s.tail(10).mean()).reset_index(name='Last10_WinPct')
    )
    out = agg.merge(last10, on=['Season','TeamID'], how='left')
    opp_ctx = out[['Season','TeamID','WinPct','Last10_WinPct']].rename(columns={
        'TeamID': 'OppID',
        'WinPct': 'Opp_WinPct',
        'Last10_WinPct': 'Opp_Last10_WinPct',
    })
    games = games.merge(opp_ctx, on=['Season','OppID'], how='left')
    sos = games.groupby(['Season','TeamID']).agg(
        OppAvgWinPct=('Opp_WinPct', 'mean'),
        OppAvgLast10WinPct=('Opp_Last10_WinPct', 'mean'),
    ).reset_index()
    out = out.merge(sos, on=['Season','TeamID'], how='left')
    out['WinPct_vs_OppAvg'] = out['WinPct'] - out['OppAvgWinPct']
    out['Last10_vs_OppAvg'] = out['Last10_WinPct'] - out['OppAvgLast10WinPct']
    return out

def build_team_seq_dict(df_scaled: pd.DataFrame, feature_cols, min_games=1):
    seq = {}
    grouped = df_scaled.groupby(['Season','TeamID'], sort=False)
    for (season, team), g in grouped:
        g = g.sort_values('DayNum')
        x = torch.tensor(g[feature_cols].values, dtype=torch.float32)
        if x.size(0) >= min_games:
            seq[(int(season), int(team))] = x
    return seq

class MatchupSeqDataset(Dataset):
    def __init__(self, pairs_df, team_seq_dict, seednum_map, winpct_map, sos_map, form_gap_map, T, feature_dim):
        self.df = pairs_df.reset_index(drop=True)
        self.team_seq = team_seq_dict
        self.seednum_map = seednum_map
        self.winpct_map = winpct_map
        self.sos_map = sos_map
        self.form_gap_map = form_gap_map
        self.T = T
        self.F = feature_dim

        keep = []
        for _, r in self.df.iterrows():
            kA = (int(r.Season), int(r.TeamA))
            kB = (int(r.Season), int(r.TeamB))
            keep.append(
                kA in team_seq_dict and
                kB in team_seq_dict and
                kA in seednum_map and
                kB in seednum_map
            )
        self.df = self.df[np.array(keep)].reset_index(drop=True)

    def _pad_trunc(self, x):
        if x.size(0) >= self.T:
            return x[-self.T:]
        return torch.cat([torch.zeros(self.T - x.size(0), self.F), x], dim=0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        season = int(r.Season)
        A = int(r.TeamA)
        B = int(r.TeamB)
        kA = (season, A)
        kB = (season, B)
        seqA = self._pad_trunc(self.team_seq[kA]).float()
        seqB = self._pad_trunc(self.team_seq[kB]).float()
        seedA = torch.tensor(self.seednum_map[kA], dtype=torch.float32)
        seedB = torch.tensor(self.seednum_map[kB], dtype=torch.float32)
        winA = torch.tensor(self.winpct_map.get(kA, 0.5), dtype=torch.float32)
        winB = torch.tensor(self.winpct_map.get(kB, 0.5), dtype=torch.float32)
        sosA = torch.tensor(self.sos_map.get(kA, 0.5), dtype=torch.float32)
        sosB = torch.tensor(self.sos_map.get(kB, 0.5), dtype=torch.float32)
        formGapA = torch.tensor(self.form_gap_map.get(kA, 0.0), dtype=torch.float32)
        formGapB = torch.tensor(self.form_gap_map.get(kB, 0.0), dtype=torch.float32)
        y = torch.tensor(float(r.y), dtype=torch.float32)
        return (
            seqA, seqB,
            seedA, seedB,
            winA, winB,
            sosA, sosB,
            formGapA, formGapB,
            y
        )

class SeqEncoder(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.rnn = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
    def forward(self, x):
        _, h = self.rnn(x)
        return h[-1]

class MatchupModel(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128, mlp_dim=128, dropout=0.25):
        super().__init__()
        self.enc = SeqEncoder(feature_dim, hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 12, mlp_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, 1)
        )
    def forward(self, seqA, seqB, seedA, seedB, winA, winB, sosA, sosB, formGapA, formGapB):
        embA = self.enc(seqA)
        embB = self.enc(seqB)
        seedA_s = 17.0 - seedA
        seedB_s = 17.0 - seedB
        extra = torch.stack([
            seedA_s, seedB_s, seedA_s - seedB_s,
            winA, winB, winA - winB,
            sosA, sosB, sosA - sosB,
            formGapA, formGapB, formGapA - formGapB,
        ], dim=1)
        x = torch.cat([embA, embB, embA - embB, embA * embB, extra], dim=1)
        return self.head(x).squeeze(1)

def brier_loss_from_logits(logits, y):
    p = torch.sigmoid(logits)
    return torch.mean((p - y) ** 2)

@torch.no_grad()
def eval_brier(model, loader, device):
    model.eval()
    total = 0.0
    n = 0
    for batch in loader:
        *inputs, y = batch
        inputs = [x.to(device) for x in inputs]
        y = y.to(device)
        logits = model(*inputs)
        loss = brier_loss_from_logits(logits, y)
        total += loss.item() * y.size(0)
        n += y.size(0)
    return total / n

def train_one_epoch(model, loader, optimizer, device, scheduler=None):
    model.train()
    total = 0.0
    n = 0
    for batch in loader:
        *inputs, y = batch
        inputs = [x.to(device, non_blocking=True) for x in inputs]
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(*inputs)
        loss = brier_loss_from_logits(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total += loss.item() * y.size(0)
        n += y.size(0)
    return total / n

@torch.no_grad()
def predict_probs(model, loader, device):
    model.eval()
    all_probs = []
    for batch in loader:
        *inputs, _ = batch
        inputs = [x.to(device) for x in inputs]
        logits = model(*inputs)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs)


In [3]:
def run_pipeline(
    prefix, 
    label, 
    team_id_min, 
    team_id_max,
    train_start_season=2009,
    val_seasons=(2023, 2024, 2025),
    pred_season=2026,
    batch_size=512, 
    epochs=30, 
    lr=3e-4, 
    early_stopping_patience=5,
    early_stopping_min_delta=1e-4,
    weight_decay=1e-4,
    out_csv='submission.csv'
    ):
    reg_path = find_file(f'{prefix}RegularSeasonDetailedResults.csv')
    tour_path = find_file(f'{prefix}NCAATourneyDetailedResults.csv')
    seeds_path = find_file(f'{prefix}NCAATourneySeeds.csv')
    massey_path = find_file_optional(f'{prefix}MasseyOrdinals.csv')
    sub_path = find_file('SampleSubmissionStage2.csv')

    seeds = pd.read_csv(seeds_path)
    tmp = seeds['Seed'].apply(parse_seed)
    seeds['Region'] = tmp.apply(lambda x: x[0])
    seeds['SeedNum'] = tmp.apply(lambda x: x[1]).astype(np.int16)
    seednum_map = {(int(r.Season), int(r.TeamID)): int(r.SeedNum) for r in seeds.itertuples()}

    def apply_seed_bump(row):
        p = row['Pred']
        season = int(row['Season'])
        seed_a = seednum_map.get((season, int(row['TeamA'])))
        seed_b = seednum_map.get((season, int(row['TeamB'])))
        if seed_a is None or seed_b is None:
            return p
        if seed_a == seed_b:
            return p
        gap = abs(seed_a - seed_b)
        if label == 'women':
            if gap >= 15:
                bump = 0.06
            elif gap >= 12:
                bump = 0.04
            elif gap >= 10:
                bump = 0.02
            else:
                bump = 0.0
        else:
            if gap >= 15:
                bump = 0.07
            elif gap >= 12:
                bump = 0.05
            elif gap >= 10:
                bump = 0.03
            else:
                bump = 0.0
        if seed_a < seed_b:
            return min(0.995, p + bump)
        return max(0.005, p - bump)

    reg = pd.read_csv(reg_path)
    W_cols = [c for c in reg.columns if c.startswith('W') & (c not in ['WTeamID', 'WLoc'])]
    L_cols = [c for c in reg.columns if c.startswith('L') & (c not in ['LTeamID', 'WLoc'])]
    id_cols = [c for c in ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WLoc', 'NumOT'] if c in reg.columns]
    reg = reg[id_cols + sorted(set(W_cols + L_cols))].copy()

    team_games = make_team_game_rows(reg)
    team_games = add_possession_efficiency_features(team_games)
    team_games = add_rolling_neteff_features_team_and_opp(team_games, windows=(5,10), feature_col='T_NetEff')

    team_games_aug = team_games.copy()
    available_systems = []
    if massey_path is not None:
        massey = pd.read_csv(massey_path)
        SYSTEMS = ['POM', 'DOK', 'MAS']
        available_systems = [s for s in SYSTEMS if s in set(massey['SystemName'].unique())]
        for system_name in available_systems:
            team_games_aug = add_team_and_opp_massey(team_games_aug, massey, system_name)

    neteff_cols = [c for c in team_games_aug.columns if 'NetEff' in c]
    team_games_aug[neteff_cols] = team_games_aug[neteff_cols].fillna(0.0)

    tour = pd.read_csv(tour_path)
    needed = ['Season','DayNum','WTeamID','LTeamID']
    for c in needed:
        if c not in tour.columns:
            raise ValueError(f'Missing {c} in tourney data')
    t1 = tour[['Season','DayNum','WTeamID','LTeamID']].copy()
    t1.rename(columns={'WTeamID':'TeamA','LTeamID':'TeamB'}, inplace=True)
    t1['y'] = 1
    t2 = tour[['Season','DayNum','WTeamID','LTeamID']].copy()
    t2.rename(columns={'WTeamID':'TeamB','LTeamID':'TeamA'}, inplace=True)
    t2['y'] = 0
    train_pairs = pd.concat([t1, t2], ignore_index=True)
    train_pairs = train_pairs.sort_values(['Season','DayNum']).reset_index(drop=True)

    wl_context = build_pre_tourney_context(reg, cutoff_day=133)
    winpct_map = {(int(r.Season), int(r.TeamID)): float(r.WinPct) for r in wl_context.itertuples()}
    sos_map = {(int(r.Season), int(r.TeamID)): float(r.OppAvgWinPct) for r in wl_context.itertuples()}
    form_gap_map = {(int(r.Season), int(r.TeamID)): float(r.Last10_vs_OppAvg) for r in wl_context.itertuples()}

    feature_cols = [
        'IsWin',
        'Score_dif',
        'OffEff_dif',
        'DefEff_dif',
        'POM_Rating_Diff',
        'DOK_Rating_Diff',
        'MAS_Rating_Diff',
        'Loc_H',
        'NetEff_Mean_dif_5',
        'NetEff_Mean_dif_10',
    ]
    for rating_col in ['POM_Rating_Diff', 'DOK_Rating_Diff', 'MAS_Rating_Diff']:
        if rating_col not in team_games_aug.columns and rating_col in feature_cols:
            feature_cols = [c for c in feature_cols if c != rating_col]

    all_seasons = sorted(train_pairs['Season'].unique())
    cv_pool = [s for s in all_seasons if (s >= train_start_season) and (s < pred_season)]
    if isinstance(val_seasons, int):
        val_seasons = [val_seasons]
    val_seasons = sorted({int(s) for s in val_seasons})
    missing_val = [s for s in val_seasons if s not in cv_pool]
    if missing_val:
        raise ValueError(f'val_seasons contain unavailable years: {missing_val}. Available CV pool: {cv_pool[0]}..{cv_pool[-1]}')

    T = 12
    F_len = len(feature_cols)

    sub = pd.read_csv(sub_path)
    def parse_id(row_id):
        season, teamA, teamB = row_id.split('_')
        return int(season), int(teamA), int(teamB)
    tmp = sub['ID'].apply(parse_id)
    sub['Season'] = tmp.apply(lambda x: x[0])
    sub['TeamA'] = tmp.apply(lambda x: x[1])
    sub['TeamB'] = tmp.apply(lambda x: x[2])

    pred_pairs = sub[sub['Season'] == pred_season][['Season','TeamA','TeamB']].copy()
    pred_pairs = pred_pairs[(pred_pairs['TeamA'] >= team_id_min) & (pred_pairs['TeamA'] <= team_id_max) &
                            (pred_pairs['TeamB'] >= team_id_min) & (pred_pairs['TeamB'] <= team_id_max)].copy()
    pred_pairs['DayNum'] = 135
    pred_pairs['y'] = 0.5

    pred_base = pred_pairs.copy()
    pred_base['ID'] = pred_base.apply(lambda r: f"{int(r.Season)}_{int(r.TeamA)}_{int(r.TeamB)}", axis=1)
    fold_pred_cols = []
    models = []

    for fold_idx, val_season in enumerate(val_seasons, start=1):
        fold_train_seasons = [s for s in cv_pool if s != val_season]
        fit_df = team_games_aug[team_games_aug['Season'].isin(fold_train_seasons)]
        if fit_df.empty:
            raise ValueError(f'fit_df is empty for fold val_season={val_season}')

        scaler = StandardScaler()
        scaler.fit(fit_df[feature_cols].values)
        team_games_scaled = team_games_aug.copy()
        team_games_scaled[feature_cols] = scaler.transform(team_games_scaled[feature_cols].values)
        team_seq = build_team_seq_dict(team_games_scaled, feature_cols, min_games=1)

        train_df = train_pairs[train_pairs['Season'].isin(fold_train_seasons)].copy()
        val_df = train_pairs[train_pairs['Season'] == val_season].copy()

        train_ds = MatchupSeqDataset(train_df, team_seq, seednum_map, winpct_map, sos_map, form_gap_map, T=T, feature_dim=F_len)
        val_ds = MatchupSeqDataset(val_df, team_seq, seednum_map, winpct_map, sos_map, form_gap_map, T=T, feature_dim=F_len)

        if len(train_ds) == 0:
            raise ValueError(f'train_ds is empty for fold val_season={val_season}; check seeds/team_seq coverage')

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = MatchupModel(F_len).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        best_val = float('inf')
        best_state = None
        best_epoch = 0
        epochs_without_improve = 0

        for epoch in range(1, epochs + 1):
            tr = train_one_epoch(model, train_loader, optimizer, device)
            va = eval_brier(model, val_loader, device) if len(val_ds) else None
            prefix_msg = f'[{label}][fold {fold_idx}/{len(val_seasons)} val={val_season}]'
            print(f'{prefix_msg} epoch {epoch}: train_brier={tr:.4f} val_brier={va:.4f}' if va is not None else f'{prefix_msg} epoch {epoch}: train_brier={tr:.4f}')

            if va is not None:
                if va < best_val - early_stopping_min_delta:
                    best_val = va
                    best_epoch = epoch
                    epochs_without_improve = 0
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    epochs_without_improve += 1
                    if epochs_without_improve >= early_stopping_patience:
                        print(f'{prefix_msg} early stopping at epoch {epoch}; best_val_brier={best_val:.4f} from epoch {best_epoch}')
                        break

        if best_state is not None:
            model.load_state_dict(best_state)
            print(f'[{label}][fold {fold_idx}/{len(val_seasons)} val={val_season}] restored best model from epoch {best_epoch} (val_brier={best_val:.4f})')

        pred_ds = MatchupSeqDataset(pred_pairs, team_seq, seednum_map, winpct_map, sos_map, form_gap_map, T=T, feature_dim=F_len)
        if len(pred_ds) == 0:
            raise ValueError(f'pred_ds is empty for fold val_season={val_season}; check pred_season, team_id range, and seed/team_seq coverage')
        pred_loader = DataLoader(pred_ds, batch_size=1024, shuffle=False, num_workers=0)

        probs = predict_probs(model, pred_loader, device)
        pred_df = pred_ds.df.copy()
        pred_df['ID'] = pred_df.apply(lambda r: f"{int(r.Season)}_{int(r.TeamA)}_{int(r.TeamB)}", axis=1)
        fold_col = f'Pred_fold_{fold_idx}'
        pred_df[fold_col] = probs
        pred_base = pred_base.merge(pred_df[['ID', fold_col]], on='ID', how='left')
        fold_pred_cols.append(fold_col)
        models.append(model)

    pred_base['Pred'] = pred_base[fold_pred_cols].mean(axis=1)
    pred_base['Pred'] = pred_base.apply(apply_seed_bump, axis=1)
    pred_df = pred_base[['ID', 'Pred']].copy()

    sub_out = sub[(sub['Season'] == pred_season) &
                  (sub['TeamA'] >= team_id_min) & (sub['TeamA'] <= team_id_max) &
                  (sub['TeamB'] >= team_id_min) & (sub['TeamB'] <= team_id_max)][['ID']].merge(
        pred_df[['ID','Pred']], on='ID', how='left'
    )
    missing = sub_out['Pred'].isna().sum()
    if missing:
        print(f'[{label}] WARNING: {missing} rows missing predictions (filled with 0.5).')
        sub_out['Pred'] = sub_out['Pred'].fillna(0.5)
    sub_out.to_csv(out_csv, index=False)
    return models, sub_out


In [ ]:
# Men
men_model, men_sub = run_pipeline(
    prefix='M', 
    label='men', 
    team_id_min=0, 
    team_id_max=2999, 
    out_csv='submission_2026_mens.csv', 
    epochs=20, 
    lr=3e-4
)

# Women
women_model, women_sub = run_pipeline(
    prefix='W', 
    label='women', 
    team_id_min=3000, 
    team_id_max=9999,
    out_csv='submission_2026_womens.csv', 
    epochs=50, 
    lr=3e-4,
    train_start_season=2004
)

# Combine
submission = pd.concat([men_sub, women_sub], ignore_index=True)
submission.to_csv('submission_2026.csv', index=False)
submission.head()


[men][fold 1/3 val=2023] epoch 1: train_brier=0.2534 val_brier=0.2455
[men][fold 1/3 val=2023] epoch 2: train_brier=0.2421 val_brier=0.2350
[men][fold 1/3 val=2023] epoch 3: train_brier=0.2293 val_brier=0.2254
[men][fold 1/3 val=2023] epoch 4: train_brier=0.2191 val_brier=0.2179
[men][fold 1/3 val=2023] epoch 5: train_brier=0.2098 val_brier=0.2121
[men][fold 1/3 val=2023] epoch 6: train_brier=0.2036 val_brier=0.2086
[men][fold 1/3 val=2023] epoch 7: train_brier=0.2005 val_brier=0.2079
[men][fold 1/3 val=2023] epoch 8: train_brier=0.1991 val_brier=0.2067
[men][fold 1/3 val=2023] epoch 9: train_brier=0.1979 val_brier=0.2072
[men][fold 1/3 val=2023] epoch 10: train_brier=0.1961 val_brier=0.2082
[men][fold 1/3 val=2023] epoch 11: train_brier=0.1948 val_brier=0.2102
[men][fold 1/3 val=2023] epoch 12: train_brier=0.1951 val_brier=0.2102
[men][fold 1/3 val=2023] epoch 13: train_brier=0.1957 val_brier=0.2109
[men][fold 1/3 val=2023] early stopping at epoch 13; best_val_brier=0.2067 from epoch 

,ID,Pred
0,2026_1101_1102,0.5
1,2026_1101_1103,0.5
2,2026_1101_1104,0.5
3,2026_1101_1105,0.5
4,2026_1101_1106,0.5
